# Filler detector, part 1: train on PodcastFillers (C4 PP2)

Trains a small model that hears "uh" and "um" in 1-second pieces of audio. Part 1 uses only the
public PodcastFillers dataset, so no student audio is involved. Part 2 (later) adapts it to the
pilot recordings and tests it on whole vivas.

**Before you start:** Runtime menu, Change runtime type, choose **T4 GPU**, Save. Your Google
Drive needs about 3 GB free.

**Steps:** Runtime, Run all. Allow Google Drive access when asked, and keep this tab open.
The first run downloads 25 GB into Colab (not your laptop), keeps about 2.5 GB in the Drive
folder `c4-filler`, and takes 1 to 2 hours. Later runs skip the download and take about 15
minutes. At the end, Drive holds `filler_cnn_v1.pt` (the model) and
`filler_cnn_v1_results.json` (its test results).

Dataset: PodcastFillers v1.0.1, https://zenodo.org/records/7121457 (Zhu, Caceres and Salamon,
"Filler Word Detection and Classification: A Dataset and Benchmark", Interspeech 2022). Free for
non-commercial research.

In [ ]:
# 1. Google Drive, GPU and libraries
!pip -q install librosa soundfile
!nvidia-smi --query-gpu=name --format=csv,noheader
from pathlib import Path

from google.colab import drive

drive.mount("/content/drive")
DRIVE = Path("/content/drive/MyDrive/c4-filler")
DRIVE.mkdir(parents=True, exist_ok=True)
DATA = DRIVE / "podcastfillers_clips.npz"

In [ ]:
# 2. Prepare the clips (first run only: 25 GB download into Colab, 1 to 1.5 hours)
import os
import shutil

import numpy as np
import pandas as pd
import soundfile as sf
from tqdm.auto import tqdm

LABELS = ["Words", "Uh", "Um", "Breath", "Laughter", "Music"]  # the dataset's six main labels
URL = "https://zenodo.org/records/7121457/files"
PARTS = ["PodcastFillers.z01", "PodcastFillers.z02", "PodcastFillers.z03", "PodcastFillers.zip"]
sh = get_ipython().system

if DATA.exists():
    print("Already in Drive, skipping the download:", DATA)
else:
    os.chdir("/content")
    free_gb = shutil.disk_usage("/content").free / 1e9
    assert free_gb > 55, f"Only {free_gb:.0f} GB free in Colab; unpacking needs 55 GB."
    for part in ["PodcastFillers.csv", *PARTS]:  # -c resumes if the cell is run again
        sh(f'wget -q -c --show-progress -O {part} "{URL}/{part}?download=1"')
    assert all(Path(p).stat().st_size > 5e9 for p in PARTS), "A download is incomplete; run this cell again."
    sh("apt-get -qq install -y zip unzip > /dev/null")
    sh("zip -q -s 0 PodcastFillers.zip --out pf_full.zip")  # join the four parts into one zip
    for part in PARTS:
        os.remove(part)
    sh('unzip -q -o pf_full.zip "*clip_wav/*" -d pf')  # only the 1-second clips
    paths = {p.name: p for p in Path("pf").rglob("*.wav")}
    assert paths, "No clips found. Run: !unzip -l /content/pf_full.zip | grep -i clip | head"
    os.remove("pf_full.zip")

    meta = pd.read_csv("PodcastFillers.csv")
    print(meta.columns.tolist())

    def col(*words):  # the CSV's column names differ from the dataset page, so find them by name parts
        found = [c for c in meta.columns if all(w in c.lower() for w in words)]
        assert found, f"No column containing {words}"
        return found[0]

    LABEL, SPLIT, CLIP, PFID = col("consolidated"), col("clip", "split"), col("clip_name"), col("pfid")
    print(meta[SPLIT].value_counts(), meta[LABEL].value_counts(), sep="\n")
    meta["label"] = meta[LABEL].astype(str).str.capitalize()
    names = meta[CLIP].astype(str)
    meta["file"] = names.where(names.str.endswith(".wav"), names + ".wav")
    meta = meta[meta[SPLIT].isin(["train", "validation", "test"]) & meta.label.isin(LABELS)]
    assert len(meta) > 70000, len(meta)
    missing = ~meta.file.isin(paths)
    print(f"{len(meta)} labelled clips, {missing.sum()} missing on disk")
    assert missing.mean() < 0.01
    meta = meta[~missing]

    X = np.zeros((len(meta), 16000), np.int16)  # 1 s at 16 kHz, stored as 16-bit to halve the size
    for i, name in enumerate(tqdm(meta.file)):
        audio, sr = sf.read(paths[name], dtype="float32", always_2d=True)
        assert sr == 16000, (name, sr)
        audio = audio.mean(1)[:16000]
        X[i, : len(audio)] = np.clip(audio, -1, 1) * 32767
    np.savez(
        "/content/clips.npz",
        x=X,
        y=meta.label.map(LABELS.index).to_numpy(np.int8),
        split=meta[SPLIT].to_numpy().astype("U10"),
        pfid=meta[PFID].astype(str).to_numpy(),
    )
    partial = DRIVE / "partial.npz"
    shutil.copy("/content/clips.npz", partial)
    partial.rename(DATA)  # only a complete copy gets the real name
    print("Saved to Drive:", DATA)

In [ ]:
# 3. Load the clips from Drive (about 1 minute)
d = np.load(DATA)
X, y, split = d["x"], d["y"].astype(np.int64), d["split"]
print(pd.crosstab(pd.Series(np.array(LABELS)[y], name="label"), pd.Series(split, name="split"), margins=True))

In [ ]:
# 4. Train (about 5 to 10 minutes on the T4 GPU)
import copy

import librosa
import torch
import torch.nn as nn
from sklearn.metrics import f1_score

torch.manual_seed(0)
np.random.seed(0)
dev = "cuda" if torch.cuda.is_available() else "cpu"
FILLERS = [LABELS.index("Uh"), LABELS.index("Um")]
EPOCHS, BATCH = 20, 256
# The live viva must compute exactly these features, so they are saved with the model.
FEATURES = {
    "sample_rate": 16000,
    "clip_samples": 16000,
    "stft": "torch.stft n_fft=400 hop=160, periodic Hann window, center=True, reflect padding, power",
    "mel": "librosa.filters.mel(sr=16000, n_fft=400, n_mels=64), stored in the model as `mel`",
    "log": "log(mel @ power + 1e-6)",
}


def block(i, o):
    return nn.Sequential(nn.Conv2d(i, o, 3, padding=1), nn.BatchNorm2d(o), nn.ReLU(), nn.MaxPool2d(2))


class FillerCNN(nn.Module):
    """1 second of 16 kHz audio in, a score for each of the six labels out (about 240k weights)."""

    def __init__(self):
        super().__init__()
        mel = librosa.filters.mel(sr=16000, n_fft=400, n_mels=64)
        self.register_buffer("mel", torch.tensor(mel, dtype=torch.float32))
        self.register_buffer("window", torch.hann_window(400))
        self.norm = nn.BatchNorm2d(1)
        self.net = nn.Sequential(
            block(1, 32), block(32, 64), block(64, 128), block(128, 128),
            nn.AdaptiveAvgPool2d(1), nn.Flatten(), nn.Dropout(0.3), nn.Linear(128, len(LABELS)),
        )

    def forward(self, wav):  # wav: (batch, 16000) floats in [-1, 1]
        power = torch.stft(wav, 400, 160, window=self.window, return_complex=True).abs() ** 2
        logmel = torch.log(self.mel @ power + 1e-6).unsqueeze(1)  # (batch, 1, 64 mels, 101 frames)
        return self.net(self.norm(logmel))


def on_device(name):
    keep = split == name
    return torch.from_numpy(X[keep]).to(dev), y[keep]


def predict(model, xs):
    model.eval()
    with torch.no_grad():
        chunks = [torch.softmax(model(xs[i : i + 1024].float() / 32768), 1).cpu() for i in range(0, len(xs), 1024)]
    return torch.cat(chunks).numpy()


def filler_f1(labels, probs):
    return f1_score(np.isin(labels, FILLERS), np.isin(probs.argmax(1), FILLERS))


Xtr, ytr = on_device("train")
Xva, yva = on_device("validation")
Xte, yte = on_device("test")
ytr_dev = torch.from_numpy(ytr).to(dev)
model = FillerCNN().to(dev)
opt = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-2)
sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, EPOCHS)
best_f1, best_epoch, best = -1.0, 0, None
for epoch in range(1, EPOCHS + 1):
    model.train()
    order = torch.randperm(len(ytr), device=dev)
    for i in range(0, len(order), BATCH):
        idx = order[i : i + BATCH]
        wav = Xtr[idx].float() / 32768
        wav = wav * 10 ** (torch.empty(len(idx), 1, device=dev).uniform_(-6, 6) / 20)  # up to 6 dB louder or quieter
        wav = torch.roll(wav, int(np.random.randint(-4000, 4001)), dims=1)  # filler up to 0.25 s off centre
        loss = nn.functional.cross_entropy(model(wav), ytr_dev[idx])
        opt.zero_grad()
        loss.backward()
        opt.step()
    sched.step()
    f1 = filler_f1(yva, predict(model, Xva))
    if f1 > best_f1:
        best_f1, best_epoch, best = f1, epoch, copy.deepcopy(model.state_dict())
    print(f"epoch {epoch:2d}  loss {loss.item():.3f}  validation filler F1 {f1:.3f}")
print(f"Best: epoch {best_epoch}, validation filler F1 {best_f1:.3f}")

In [ ]:
# 5. Test on clips the model has never seen
from sklearn.metrics import classification_report, confusion_matrix, precision_recall_fscore_support

model.load_state_dict(best)
pred = predict(model, Xte).argmax(1)
print(classification_report(yte, pred, labels=range(len(LABELS)), target_names=LABELS, digits=3))
print("Rows: true label, columns: predicted")
print(pd.DataFrame(confusion_matrix(yte, pred), index=LABELS, columns=LABELS))
p, r, f, _ = precision_recall_fscore_support(np.isin(yte, FILLERS), np.isin(pred, FILLERS), average="binary")
print(f"\nFiller (uh or um) against everything else: precision {p:.3f}, recall {r:.3f}, F1 {f:.3f}")

In [ ]:
# 6. Save the model and its results to Drive
import datetime
import hashlib
import json
from zoneinfo import ZoneInfo

ckpt = DRIVE / "filler_cnn_v1.pt"
torch.save({"state_dict": best, "labels": LABELS, "fillers": FILLERS, "features": FEATURES}, ckpt)
results = {
    "model": ckpt.name,
    "sha256": hashlib.sha256(ckpt.read_bytes()).hexdigest(),
    "trained_at": datetime.datetime.now(ZoneInfo("Asia/Colombo")).isoformat(timespec="seconds"),
    "dataset": "PodcastFillers v1.0.1, doi:10.5281/zenodo.7121457, consolidated labels, official clip splits",
    "clips": {s: int((split == s).sum()) for s in ["train", "validation", "test"]},
    "epochs": EPOCHS,
    "best_epoch": best_epoch,
    "seed": 0,
    "torch": torch.__version__,
    "gpu": torch.cuda.get_device_name() if dev == "cuda" else "cpu",
    "validation_filler_f1": best_f1,
    "test_filler": {"precision": p, "recall": r, "f1": f},
    "test_report": classification_report(
        yte, pred, labels=range(len(LABELS)), target_names=LABELS, output_dict=True
    ),
}
(DRIVE / "filler_cnn_v1_results.json").write_text(json.dumps(results, indent=2, default=float))
drive.flush_and_unmount()  # makes sure Drive has both files before you close the tab
print(json.dumps({k: results[k] for k in ["model", "sha256", "test_filler"]}, indent=2, default=float))

**Done? Keep the training log for your supervisors before closing anything:**
1. Press Ctrl+S (saves this notebook with all its output in Drive, folder Colab Notebooks).
2. File, Download, Download .ipynb (an offline copy with the output).
3. Optional: File, Print, Save as PDF.

What to show them: cell 2 (data source and counts), cell 4 (the training log), cell 5 (test
results on unseen clips) and `filler_cnn_v1_results.json` in Drive (date, scores, model
fingerprint).

**Send the printed results of cells 4 to 6 to Claude.** Then: Runtime, Disconnect and delete
runtime (the 25 GB download is removed; Drive keeps the clips and the model).

If something fails:
- "Only N GB free": Runtime, Disconnect and delete runtime, then Run all again.
- A download stops: run cell 2 again; it continues where it stopped.
- "No clips found": run the command in the message and send the output to Claude.
- No GPU available: try again later; training on the CPU is much slower.

Next, part 2: adapt the model to the pilot recordings (P01 to P10, with the CrisperWhisper filler
times), test it on whole vivas, then freeze it before the first new student.